In [1]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import jaccard

import metric_utils

import json
from constants import Annotations, AnnotatedReport
import time
from IPython.display import clear_output

from huggingface_hub import login

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, DefaultDataCollator
from datasets import load_dataset, Dataset, DatasetDict

from model_utils import create_label_to_id_map

from train_utils import get_device, add_nan_flag_to_df, create_list_of_annotated_reports, create_hugging_face_dataset, add_target_columns_to_dataset, get_normalization_stats
from classifiers import ReportExtractor
from tqdm import tqdm
import sklearn.metrics as metrics
import numpy as np
from pprint import pprint
import json

In [2]:
############
# Parameters
############
# Data parameters
TRAIN_FILE_NAME = "train_split.csv"
VALIDATION_FILE_NAME = "validation_split.csv"
TEST_FILE_NAME = "test_split.csv"

In [3]:
plt.style.use('ggplot')

In [4]:
base_dir = Path.cwd().parent

In [5]:
# Set device
device = get_device()
print(f'{device = }')

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
NVIDIA GeForce GTX 1060 6GB
device = device(type='cuda')


In [6]:
model = ReportExtractor.from_pretrained(base_dir / "saved_models" / "report_extractor", device=device)
model.to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(base_dir / "saved_models" / "report_extractor")

Some weights of BertModel were not initialized from the model checkpoint at IVN-RIN/bioBIT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
###########
# Load data
###########
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
    'test': TEST_FILE_NAME
}
paths = {split: base_dir / "data" / file_name
         for split, file_name in file_names.items()}
data = {split: pd.read_csv(path) for split, path in paths.items()}
train_data, validation_data, test_data = data['train'], data['validation'], data['test']
# Log
print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")
# Create nan-flag columns
for split, df in data.items():
    add_nan_flag_to_df(df)
# Create lists of Annotated reports
annotated_reports =  {split: create_list_of_annotated_reports(data[split]) for split in file_names}

len(train_data) = 167
len(validation_data) = 46
len(test_data) = 56


In [13]:
annotated_reports['train'][0].report_data.model_dump()

{'morfologia': 'solido_polipoide',
 'posizione': ['basso'],
 'ore_inizio': None,
 'ore_inizio_is_nan': True,
 'ore_fine': None,
 'ore_fine_is_nan': True,
 'spessore_parietale': None,
 'spessore_parietale_is_nan': True,
 'estensione_cranio_caudale': 50,
 'estensione_cranio_caudale_is_nan': False,
 'distanza_oai': None,
 'distanza_oai_is_nan': True,
 'riflessione_peritoneale_anteriore': None,
 'infiltrazione_tessuto_adiposo': 'no',
 'infiltrazione_sfinteri': None,
 'infiltrazione_organi_extra': None,
 'infiltrazione_organi_dettagli': [],
 'coinvolgimento_riflessione_peritoneale': None,
 'coinvolgimento_fascia_mesorettale': None,
 'linfonodi_sospetti': 0,
 'numero_linfonodi_non_conosciuto': True,
 'sedi_linfonodi': ['mesorettali'],
 'depositi_tumorali': None,
 'numero_depositi': 0,
 'emvi_esteso': 'no',
 'stadio_T': 'T1-2',
 'stadio_N': 'N+',
 'stadio_N1c': False,
 'mrf': None,
 'emvi': None,
 'metastasi': None}

In [8]:
# Check the maximum number of tokens for each report
print('model context length:', model.encoder.config.max_position_embeddings)
for split in annotated_reports:
    print(f'{split}: {len(annotated_reports[split])} reports')
    max_n_tokens = 0
    del_list = []
    for i, report in enumerate(annotated_reports[split]):
        x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
        max_n_tokens = max(max_n_tokens, x)
        if x > model.encoder.config.max_position_embeddings:
            del_list.append(i)
    print(del_list)
    print(f'{max_n_tokens = }')
    # Delete long reports
    for i in del_list[::-1]:
        annotated_reports[split].pop(i)
    print(f'After deletion: {len(annotated_reports[split])} reports')

Token indices sequence length is longer than the specified maximum sequence length for this model (1427 > 512). Running this sequence through the model will result in indexing errors


model context length: 512
train: 167 reports
[0, 2, 10, 25, 30, 33, 38, 77, 91, 93, 97, 98, 99, 115, 116, 127, 130, 134, 148, 149, 150, 158, 161, 162, 163, 164, 166]
max_n_tokens = 1427
After deletion: 140 reports
validation: 46 reports
[7, 11, 23, 33, 34, 37, 38, 42, 44]
max_n_tokens = 1008
After deletion: 37 reports
test: 56 reports
[2, 6, 16, 23, 26, 28, 33, 37, 42, 43, 45, 47]
max_n_tokens = 861
After deletion: 44 reports


In [9]:
##############################
# Create Hugging Face Datasets
##############################
dataset = DatasetDict({
    split: create_hugging_face_dataset(annotated_reports[split])
    for split in annotated_reports
})
# Tokenize text and set format to torch
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="longest", max_length=model.encoder.config.max_position_embeddings)
dataset = dataset.map(tokenize_function, batched=True)
cols_to_remove = [col for col in ["token_type_ids"] if col in dataset['validation'].column_names]
dataset = dataset.remove_columns(cols_to_remove)
dataset.set_format('torch')
# Add annotation fields to the dataset
for split, reports in annotated_reports.items():
    dataset[split] = add_target_columns_to_dataset(dataset=dataset[split],
                                                   annotated_reports=reports,
                                                   label_to_id_map=model.label_to_id_map,
                                                   classification_columns=model.classification_fields,
                                                   binary_classification_columns=model.binary_classification_fields,
                                                   multiple_choice_columns=model.multiple_choice_fields,
                                                   regression_columns=model.regression_fields,
                                                   normalization_stats=model.normalization_stats)


Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

In [10]:
dataset.set_format(
    type="torch",
)
loader = DataLoader(dataset["test"], batch_size=16, shuffle=False)
model.eval()
batches_results = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        model_output = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        reworked_output = model.rework_output(model_output)

        batches_results.append(reworked_output)

    predictions = {
        field: np.concatenate([r[field] for r in batches_results])
        for field in model.all_fields()
    }

In [55]:
actual = dict()
for f in model.all_fields():
    if f in model.regression_fields:
        mu, std = model.normalization_stats[f]
        actual[f] = np.round(np.abs((dataset['test'][f] * std) + mu), 2)
    else:
        actual[f] = dataset['test'][f]

In [57]:
campo = 'distanza_oai_is_nan'
print(actual[campo][:5])
print(predictions[campo][:5])

[0 0 1 1 0]
[0.25875017 0.30740753 0.25613827 0.26141945 0.2446874 ]


In [54]:
campo = 'ore_inizio'
dataset.set_format('numpy')

pred_prob = predictions[campo][:5]
y_true = dataset['test'][campo][:5]

print(pred_prob)
print(y_true)
mu, std = model.normalization_stats[campo]

[10.027817  9.750075  9.959853 10.419493 10.010245]
[-2.948685 -2.948685 -2.948685 -2.948685 -2.948685]


In [39]:
mu, std = model.normalization_stats['ore_inizio']
np.abs(np.round(dataset['test']['ore_inizio'] * std + mu))

array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        6.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 12., 12., 12.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 12.,  3.,  0.,
        0.,  1.,  3.,  0., 12.])

In [40]:
predictions['ore_inizio']

array([10.027817 ,  9.750075 ,  9.959853 , 10.419493 , 10.010245 ,
        9.763863 ,  9.174756 ,  9.871873 ,  9.677256 ,  9.855339 ,
       10.222757 ,  9.690041 ,  9.761908 ,  9.956261 , 10.050973 ,
        9.6739025,  9.680229 ,  9.217694 ,  9.688355 ,  9.735561 ,
        9.760233 ,  9.8472185,  9.849707 ,  9.598262 ,  9.8941145,
        9.431718 ,  9.15099  ,  9.771431 , 10.20807  ,  9.845375 ,
       10.102288 ,  9.840214 , 10.133766 , 10.262039 ,  9.471149 ,
        9.673344 ,  9.343577 , 10.081224 ,  9.887465 ,  9.489399 ,
        9.624778 , 10.092763 , 10.184837 ,  9.785534 ], dtype=float32)